# Escalation after the agreement rule
Step 1 (6 queries): answer if inliers ≥ 8 or ≥ 2 queries agree within 1σ (each with ≥ 3 inliers).
Step 2: photos still skipped are re-run with 27 queries (9 headings −10…+10 × scales 0.95, 1.0, 1.05; dumps `dump_esc27_*`).
The escalated decision picks the most inliers + z over all 27 queries and answers at ≥ 8 inliers or ≥ k agreeing queries.
More queries make chance agreement more likely, so k = 2, 3, 4 are all scored. Wrong answers matter most on NUDT.

In [1]:
import sys, glob
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not ((ROOT / "src").is_dir() and (ROOT / "train/checks").is_dir()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from train.checks.vote_decision import load_photo, choose_max, query_support, error_sigma
from train.checks.decision_gate import points_from_error
R = ROOT / "tests/Test_Iteration_3/results/fresh_run"

def first_step(name):
    """Per photo from the 6-query dump (or the gate-row csv for NUDT seed 101): error, inliers, agreement."""
    if name == "nudt101":
        g = pd.read_csv(R / "gate_rows2_nudt_conf0.22.csv")
        return pd.DataFrame(dict(photo=g.photo, err=g.error_sigma, inl=g.inliers, agree=g.agree_queries))
    rows = []
    for f in glob.glob(str(R / ("dump_%s_conf0.22" % name) / "photo_*.npz")):
        P = load_photo(f); c = P["cand"]; k = choose_max(c); k = k if k >= 0 else int(np.argmax(c["inliers"]))
        rows.append(dict(photo=int(Path(f).stem.split("_")[1]), err=error_sigma(c["x"][k], c["y"][k], P["truth"], P["sigma"]),
                         inl=c["inliers"][k], agree=query_support(c, c["x"][k], c["y"][k], P["sigma"])))
    return pd.DataFrame(rows)

def escalated(tag):
    rows = []
    for f in glob.glob(str(R / ("dump_esc27_%s_conf0.22" % tag) / "photo_*.npz")):
        P = load_photo(f); c = P["cand"]; k = choose_max(c)
        if k < 0:
            rows.append(dict(photo=int(Path(f).stem.split("_")[1]), err2=np.inf, inl2=0, agree2=0, heading2=np.nan)); continue
        rows.append(dict(photo=int(Path(f).stem.split("_")[1]), err2=error_sigma(c["x"][k], c["y"][k], P["truth"], P["sigma"]),
                         inl2=c["inliers"][k], agree2=query_support(c, c["x"][k], c["y"][k], P["sigma"]), heading2=c["heading"][k]))
    return pd.DataFrame(rows)

SETS = {"official": "official", "nudt101": "nudt101", "nudt202": "nudt_seed202"}
results = []
for tag, dump in SETS.items():
    t = first_step(dump if tag != "nudt101" else "nudt101").merge(escalated(tag), on="photo", how="left")
    pts1 = points_from_error(t.err.values)
    bar8 = t.inl.values >= 8
    step1 = bar8 | (t.agree.values >= 2)
    def score(f, p): return np.where(f, p, 0).sum() / len(t) * 6
    def wrong(f, p): return int((f & (p == -20)).sum())
    results.append(dict(set=tag, n=len(t), escalated=int(t.err2.notna().sum()), variant="bar 8", score=score(bar8, pts1), wrong=wrong(bar8, pts1)))
    results.append(dict(set=tag, n=len(t), escalated=int(t.err2.notna().sum()), variant="agree>=2", score=score(step1, pts1), wrong=wrong(step1, pts1)))
    esc = t.err2.notna().values
    pts2 = np.where(esc, points_from_error(np.nan_to_num(t.err2.values, posinf=1e9)), pts1)
    for k in [2, 3, 4]:
        f2 = step1 | (esc & ((t.inl2.values >= 8) | (t.agree2.values >= k)))
        pts = np.where(step1, pts1, pts2)
        results.append(dict(set=tag, n=len(t), escalated=int(esc.sum()), variant="agree>=2 + esc27 (bar 8 or agree>=%d)" % k, score=score(f2, pts), wrong=wrong(f2, pts)))
    for bar2 in [10, 11, 12, 15]:
        f2 = step1 | (esc & (t.inl2.values >= bar2))
        pts = np.where(step1, pts1, pts2)
        results.append(dict(set=tag, n=len(t), escalated=int(esc.sum()), variant="agree>=2 + esc27 (bar %d only)" % bar2, score=score(f2, pts), wrong=wrong(f2, pts)))
    print(tag); display(t[esc].sort_values("inl2", ascending=False).head(25).round(2))
display(pd.DataFrame(results).round(2))

official


,photo,err,inl,agree,err2,inl2,agree2,heading2
279,148,624.34,0.0,0,0.65,45.0,7.0,-10.0
204,91,617.19,3.0,1,0.85,26.0,6.0,-10.0
269,245,0.38,6.0,1,0.49,17.0,9.0,2.5
356,33,444.81,0.0,0,0.41,15.0,8.0,7.5
70,66,0.22,2.0,0,0.55,10.0,4.0,-2.5
142,99,1.65,6.0,1,0.26,10.0,3.0,-5.0
295,172,0.39,4.0,1,0.33,9.0,6.0,10.0
318,347,189.46,7.0,1,0.61,8.0,6.0,-2.5
235,222,0.67,7.0,1,0.67,7.0,4.0,-5.0
311,210,0.29,6.0,1,0.39,7.0,5.0,-2.5


nudt101


,photo,err,inl,agree,err2,inl2,agree2,heading2
65,114,0.84,0,0,0.39,97.0,6.0,-10.0
155,275,266.06,3,1,0.42,49.0,6.0,-10.0
57,101,33.54,4,1,0.62,24.0,5.0,-5.0
5,13,50.84,5,1,0.68,23.0,9.0,0.0
144,248,805.48,7,1,1.74,19.0,1.0,-2.5
33,58,1021.98,0,0,0.38,17.0,1.0,-2.5
28,49,810.44,0,0,0.51,13.0,1.0,2.5
99,164,1.00,6,1,0.17,11.0,4.0,-2.5
272,451,1.45,0,0,0.93,10.0,3.0,-2.5
87,143,110.76,7,1,110.61,8.0,2.0,-5.0


nudt202


,photo,err,inl,agree,err2,inl2,agree2,heading2
103,204,389.93,7.0,1,0.46,23.0,2.0,7.5
114,76,384.44,0.0,0,0.19,20.0,1.0,10.0
44,377,32.66,3.0,1,0.98,15.0,1.0,-7.5
181,173,86.70,0.0,0,0.78,15.0,2.0,-7.5
104,102,68.90,0.0,0,0.52,14.0,7.0,-2.5
1,249,72.69,0.0,0,1.12,13.0,3.0,2.5
110,263,0.40,5.0,1,363.94,10.0,6.0,10.0
52,281,1.41,5.0,1,116.21,8.0,6.0,5.0
63,435,1.94,5.0,1,0.60,8.0,3.0,0.0
27,280,753.94,0.0,0,737.75,4.0,1.0,-2.5


,set,n,escalated,variant,score,wrong
0,official,379,38,bar 8,49.01,0
1,official,379,38,agree>=2,52.91,0
2,official,379,38,agree>=2 + esc27 (bar 8 or agree>=2),56.07,0
3,official,379,38,agree>=2 + esc27 (bar 8 or agree>=3),55.28,0
4,official,379,38,agree>=2 + esc27 (bar 8 or agree>=4),54.81,0
5,official,379,38,agree>=2 + esc27 (bar 10 only),53.86,0
6,official,379,38,agree>=2 + esc27 (bar 11 only),53.54,0
7,official,379,38,agree>=2 + esc27 (bar 12 only),53.54,0
8,official,379,38,agree>=2 + esc27 (bar 15 only),53.54,0
9,nudt101,300,29,bar 8,53.00,0


**Why the escalated bar must be higher.** With 27 queries the decision looks at 270 candidates instead of 60, so the best false place gets more chances to collect inliers. On the escalated NUDT photos the strongest false place reached 8 or more inliers on 12 % of photos (max 10); on the 6-query runs the official max false count was 14 but the true place nearly always beat it. Agreement among the 27 queries is also not independent evidence: neighbouring headings 2.5° apart see almost the same image and land on the same false place (NUDT wrong answers with 6 agreeing queries).